# TRACE-Seg3D – ISBI pipeline trên Colab

Chạy **lần lượt từ trên xuống**. Mọi thứ nặng (data đã tiền xử lý, split, checkpoint, kết quả) được lưu vào **Google Drive**, nên khi Colab ngắt kết nối thì chỉ cần chạy lại các cell: bước nào xong rồi sẽ được bỏ qua.

Cần chuẩn bị:
1. File code `TRACE-Seg3D_ISBI.zip` (bản mới nhất).
2. UTSW-Glioma đặt trên Drive (đổi đường dẫn ở cell 4). Chưa có UTSW thì vẫn chạy được phần BraTS.

## 1. Drive + GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
WORK = '/content/drive/MyDrive/trace_isbi'          # nơi lưu data đã xử lý / split / run / kết quả
os.makedirs(WORK, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

## 2. Code

In [ ]:
from google.colab import files
uploaded = files.upload()          # chọn TRACE-Seg3D_ISBI.zip
zip_name = next(iter(uploaded))
!rm -rf /content/Counterfactual-Representation
!unzip -q "/content/{zip_name}" -d /content/
%cd /content/Counterfactual-Representation
!pip install -q -e .
import sys; sys.path[:0] = ['/content/Counterfactual-Representation', '/content/Counterfactual-Representation/src']
os.environ['PYTHONPATH'] = '.:src'

In [ ]:
# kiểm tra nhanh code chạy được trên máy này (~30 s)
!PYTHONPATH=.:src python -m pytest -q tests/test_trace_seg3d.py 2>&1 | tail -2

## 3. BraTS 2020 (bản NIfTI chính thức trong dataset Kaggle `awsaf49/brats20-dataset-training-validation`)

In [ ]:
import kagglehub, glob
path = kagglehub.dataset_download('awsaf49/brats20-dataset-training-validation')
hits = glob.glob(f'{path}/**/MICCAI_BraTS2020_TrainingData', recursive=True)
assert hits, f'Không tìm thấy MICCAI_BraTS2020_TrainingData trong {path}'
BRATS_ROOT = hits[0]
print('BRATS_ROOT =', BRATS_ROOT)
print('số case:', len(glob.glob(f'{BRATS_ROOT}/BraTS20_Training_*')), '| name_mapping.csv:', os.path.exists(f'{BRATS_ROOT}/name_mapping.csv'))

## 4. UTSW-Glioma (đặt trên Drive) – **sửa 2 đường dẫn này**

In [ ]:
UTSW_ROOT = '/content/drive/MyDrive/data/PKG - UTSW-Glioma/UTSW-Glioma'   # thư mục chứa BT0001, BT0002, ...
UTSW_META = '/content/drive/MyDrive/data/UTSW_Glioma_Metadata-2-1.tsv'
HAS_UTSW = os.path.isdir(UTSW_ROOT)
print('UTSW có sẵn:', HAS_UTSW, '| số case:', len(glob.glob(f'{UTSW_ROOT}/BT*')) if HAS_UTSW else 0, '| metadata:', os.path.exists(UTSW_META))
if not HAS_UTSW:
    print('-> chưa có UTSW: notebook vẫn chạy BraTS (ID), phần OOD sẽ bị bỏ qua cho tới khi có UTSW.')

## 5. Tiền xử lý (chạy 1 lần; kết quả lưu trên Drive)
Colab chỉ có 2 CPU nên BraTS mất khoảng 30–60 phút. Nếu bị ngắt giữa chừng, chạy lại cell này: các case đã xử lý xong sẽ được bỏ qua.

In [ ]:
PROC_BRATS = f'{WORK}/processed/brats_128'
PROC_UTSW  = f'{WORK}/processed/utsw_128'
!python -m trace_seg3d.preprocess brats-nifti --root "{BRATS_ROOT}" --out "{PROC_BRATS}" --workers 2
if HAS_UTSW:
    meta = f'--metadata "{UTSW_META}"' if os.path.exists(UTSW_META) else ''
    !python -m trace_seg3d.preprocess utsw --root "{UTSW_ROOT}" {meta} --out "{PROC_UTSW}" --workers 2

## 6. Kiểm tra data – **bắt buộc xem hình trước khi train**
Điều kiện: dòng `T1CE > T1 + 0.5 inside ET ... OK` phải xuất hiện (FLAIR sáng ngang T1CE trong ET là bình thường), và các case của hai dataset phải cùng chiều khi nhìn cùng một view.

In [ ]:
from IPython.display import Image, display
dirs = f'"{PROC_BRATS}"' + (f' "{PROC_UTSW}"' if HAS_UTSW else '')
!python -m trace_seg3d.check_data {dirs} --png "{WORK}/check.png"
display(Image(f'{WORK}/check.png'))

## 7. Split cố định (tạo 1 lần, cả nhóm dùng chung file này)

In [ ]:
SPLIT_BRATS = f'{WORK}/splits/brats.json'
SPLIT_UTSW  = f'{WORK}/splits/utsw.json'
if not os.path.exists(SPLIT_BRATS):
    !python -m trace_seg3d.splits --index "{PROC_BRATS}/index.csv" --out "{SPLIT_BRATS}"
if HAS_UTSW and not os.path.exists(SPLIT_UTSW):
    !python -m trace_seg3d.splits --index "{PROC_UTSW}/index.csv" --out "{SPLIT_UTSW}" --exclude-auto-labels
import json
for name, f in (('brats', SPLIT_BRATS), ('utsw', SPLIT_UTSW)):
    if os.path.exists(f):
        d = json.load(open(f))
        print(name, {k: len(v) for k, v in d.items() if not k.startswith('_')})

## 7b. Copy data đã xử lý từ Drive sang ổ local (chạy **mỗi phiên**)
Nếu đọc data trực tiếp từ Drive trong lúc train thì sẽ rất chậm.

In [ ]:
LOCAL = '/content/processed'
!mkdir -p {LOCAL}
!rsync -a --info=progress2 "{PROC_BRATS}" {LOCAL}/
if HAS_UTSW:
    !rsync -a --info=progress2 "{PROC_UTSW}" {LOCAL}/
DATA_BRATS_LOCAL = f'{LOCAL}/brats_128'
DATA_UTSW_LOCAL = f'{LOCAL}/utsw_128'


## 8. Chọn PATCH/BATCH vừa VRAM và đo tốc độ (khoảng 10–20 phút)
Cell này tự thử các cấu hình từ lớn tới nhỏ (mode `trace`, nặng nhất) và giữ cấu hình lớn nhất không bị out of memory. Trên T4, 112³ × batch 2 **không vừa**.

In [ ]:
PATCH, BATCH = 96, 2   # đã đo trên T4: 112x2 out of memory, 96x2 dùng ~13 GB (trace). Bỏ qua phần dưới nếu không cần đo lại.
RUN_TIMING = False
if RUN_TIMING:
    import subprocess, json, os
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    CANDIDATES = [(112, 2), (96, 2), (112, 1), (96, 1), (80, 2)]   # (patch, batch), lớn -> nhỏ
    PATCH = BATCH = None
    for patch, batch in CANDIDATES:
        out = f'/content/timing_run_{patch}x{batch}'
        subprocess.run(['rm', '-rf', out])
        cmd = ['python', '-m', 'trace_seg3d.train', '--data-dir', DATA_BRATS_LOCAL, '--splits', SPLIT_BRATS,
               '--mode', 'trace', '--out', out, '--epochs', '1', '--cct-start-epoch', '0', '--val-every', '1',
               '--patch-size', str(patch), '--batch-size', str(batch), '--amp', '--workers', '2',
               '--max-train-cases', '20', '--max-val-cases', '2']
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f'{out}/history.json'):
            PATCH, BATCH = patch, batch
            break
        oom = 'OutOfMemoryError' in r.stderr or 'out of memory' in r.stderr
        print(f'{patch}^3 x {batch}: ' + ('out of memory -> thử cấu hình nhỏ hơn' if oom else 'LỖI KHÁC:\n' + r.stderr[-3000:]))
        if not oom:
            break
    assert PATCH, 'Không cấu hình nào chạy được – gửi output cell này cho Claude.'
    h = json.load(open(f'/content/timing_run_{PATCH}x{BATCH}/history.json'))[-1]
    n_train = len(json.load(open(SPLIT_BRATS))['train'])
    per_epoch_min = h['time_s'] / 20 * n_train / 60
    print(f"==> chọn PATCH={PATCH}, BATCH={BATCH} | VRAM peak {h.get('gpu_peak_gb')} GB | "
          f"~{per_epoch_min:.1f} phút/epoch trên toàn bộ BraTS train (mode trace; baseline nhanh hơn ~1.5x)")
    for e in (40, 60, 80):
        print(f'   EPOCHS={e}: ~{e*per_epoch_min/60:.1f} giờ cho 1 run trace')

## 9. Chạy thí nghiệm

Ngân sách GPU là giới hạn chính: mỗi run tốn khoảng `EPOCHS × phút/epoch`, và baseline nhanh hơn trace khoảng 1,5 lần.

Thứ tự ưu tiên (mỗi bước chạy trong một hoặc nhiều phiên Colab; kết quả lưu trên Drive nên nối tiếp được):
* **A.** 1 seed cho `baseline styleaug trace`, ở mọi nguồn đang có → đủ cho bảng chính sơ bộ và bảng audit.
* **B.** Thêm seed 1, 2 cho 3 phương pháp trên.
* **C.** Ablation (`trace_nocct trace_nostab trace_noproxy`), 1 seed, chỉ chạy cho nguồn UTSW.

Đổi các biến ở cell dưới cho từng bước rồi chạy lại.

Nếu Colab ngắt giữa chừng: chạy lại từ cell 1 (bỏ qua upload nếu code không đổi), cell 7b, cell 8 có thể bỏ qua bằng cách gán tay `PATCH, BATCH = 96, 2`, rồi chạy cell 9. Run đang dở sẽ **tiếp tục từ epoch cuối** (file `resume.pt`), không train lại từ đầu.

In [ ]:
os.environ.update({
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'PYTHON_BIN': 'python',
    'DATA_BRATS': DATA_BRATS_LOCAL, 'SPLIT_BRATS': SPLIT_BRATS,
    'DATA_UTSW': DATA_UTSW_LOCAL, 'SPLIT_UTSW': SPLIT_UTSW,
    'RUNS': f'{WORK}/runs', 'RESULTS': f'{WORK}/results',
    'SOURCES': 'utsw brats',
    'MAIN_METHODS': 'baseline styleaug trace',
    'SEEDS': '0',            # bước B: '0 1 2'
    'ABLATIONS': '',         # bước C: 'trace_nocct trace_nostab trace_noproxy'  (và SOURCES='utsw')
    'ABL_SEEDS': '0',
    'EPOCHS': '80',          # chọn theo thời gian đo ở cell 8
    'WORKERS': '2', 'AMP': '1',
    'EXTRA_TRAIN_ARGS': f'--patch-size {PATCH} --batch-size {BATCH} --val-every 5',
})

In [ ]:
!bash scripts/isbi/run_all.sh 2>&1 | grep -v FutureWarning | grep -v "warnings.warn" 

## 10. Xem và tải kết quả

In [ ]:
!cat "{WORK}/results/final/table.md"
!ls "{WORK}/results" | head -50

In [ ]:
!cd "{WORK}" && zip -qr /content/results_isbi.zip results
from google.colab import files
files.download('/content/results_isbi.zip')